# Pipeline Tiền xử lý dữ liệu: Cigarette Smoker Detection
 Notebook này thực hiện các bước làm sạch và chuẩn hóa dữ liệu dựa trên kết quả EDA:<br>
1. Loại bỏ ảnh hỏng & trùng lặp: Đảm bảo tính toàn vẹn của tập dữ liệu.<br>
2. Kiểm tra và Lọc nhiễu (Noise Estimation)<br>
3. Tăng cường độ tương phản cục bộ, thay đổi kích thước nhưng giữ nguyên tỷ lệ ảnh để không làm biến dạng điếu thuốc.<br>
4. Thực thi lưu trữ dữ liệu đã xử lý<br>
5. Chia tập dữ liệu (Split Train/Val/Test)<br>
6. Tăng cường dữ liệu (Offline Data Augmentation)<br>


In [ ]:
import os
import glob
import cv2
import numpy as np
import albumentations as A
from skimage.restoration import estimate_sigma
import seaborn as sns
import splitfolders
import hashlib
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import shutil

## Thiết lập đường dẫn và tổng thể về lớp (Classes)

In [ ]:
import os
# Cố định đường dẫn tuyệt đối (Absolute Path) để không bị lỗi folder lưu lung tung
RAW_DATASET_DIR = r'D:\Dev\XLA\smoking_detection_ai\data\dataset\data'
PROCESSED_DATASET_DIR = r'D:\Dev\XLA\smoking_detection_ai\data\processed_dataset'

CLASSES = ['smoking', 'not_smoking']
TARGET_SIZE = (224, 224)

In [ ]:
for cls in CLASSES:
    os.makedirs(os.path.join(PROCESSED_DATASET_DIR, cls), exist_ok=True)

In [ ]:
def get_image_paths(directory):
    paths = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.JPG', '*.JPEG'):
        paths.extend(glob.glob(os.path.join(directory, "**", ext), recursive=True))
    unique_paths = list(set(paths))
    return unique_paths

## 1. Làm sạch dữ liệu (Deduplication)
Chúng ta sẽ quét qua toàn bộ thư mục để loại bỏ các file không phải định dạng ảnh hoặc các file có nội dung giống hệt nhau (tránh Overfitting).

In [ ]:
hashes = set()
valid_samples = []
stats = {'corrupt': 0, 'duplicate': 0, 'valid': 0}

for cls in CLASSES:
    cls_dir = os.path.join(RAW_DATASET_DIR, cls)
    if not os.path.exists(cls_dir):
        print(f"Cảnh báo: Không tìm thấy thư mục {cls_dir}")
        continue

    paths = get_image_paths(cls_dir)
    for path in tqdm(paths, desc=f"Đang quét lớp {cls}"):
        try:
            # Check lỗi header ảnh
            with Image.open(path) as img:
                img.verify()

            # Check trùng lặp bằng MD5
            with open(path, 'rb') as f:
                f_hash = hashlib.md5(f.read()).hexdigest()

            if f_hash in hashes:
                stats['duplicate'] += 1
                continue

            hashes.add(f_hash)
            valid_samples.append((path, cls))
            stats['valid'] += 1

        except Exception:
            stats['corrupt'] += 1

print(f"Thống kê: {stats['valid']} ảnh hợp lệ, {stats['duplicate']} ảnh trùng, {stats['corrupt']} ảnh hỏng.")

In [ ]:
def check_and_visualize_duplicates(raw_dir, classes, num_pairs=5):
    hashes = {} # Lưu {hash_value: path_gốc}
    duplicate_pairs = [] # Lưu danh sách tuple (path_trùng, path_gốc, class)

    print("Đang đối chiếu lại mã Hash để tìm các cặp ảnh trùng lặp...")
    for cls in classes:
        cls_dir = os.path.join(raw_dir, cls)
        paths = get_image_paths(cls_dir)

        for path in paths:
            try:
                with open(path, 'rb') as f:
                    f_hash = hashlib.md5(f.read()).hexdigest()

                if f_hash in hashes:
                    # Nếu hash đã tồn tại -> Đây là bản sao
                    duplicate_pairs.append((path, hashes[f_hash], cls))
                else:
                    # Nếu chưa tồn tại -> Lưu làm bản gốc đầu tiên
                    hashes[f_hash] = path
            except Exception:
                continue

    if not duplicate_pairs:
        print("Không tìm thấy cặp trùng lặp nào.")
        return

    print(f"Tổng số ảnh bản sao tìm thấy: {len(duplicate_pairs)}")

    # Thiết lập khung hiển thị
    num_show = min(num_pairs, len(duplicate_pairs))
    fig, axes = plt.subplots(num_show, 2, figsize=(12, 5 * num_show))

    # Sửa lỗi index mảng nếu chỉ in 1 hàng
    if num_show == 1:
        axes = axes.reshape(1, 2)

    import random
    # Lấy ngẫu nhiên các cặp để xem cho khách quan
    sample_pairs = random.sample(duplicate_pairs, num_show)

    for i in range(num_show):
        path_dup, path_orig, cls_name = sample_pairs[i]

        img_dup = Image.open(path_dup)
        img_orig = Image.open(path_orig)

        # Cột 1: Hiển thị bản sao (File bị xóa)
        axes[i, 0].imshow(img_dup)
        axes[i, 0].set_title(f"BẢN SAO (Bị loại trừ): {os.path.basename(path_dup)}\nLớp: {cls_name}")
        axes[i, 0].axis('off')

        # Cột 2: Hiển thị bản gốc (File được giữ lại)
        axes[i, 1].imshow(img_orig)
        axes[i, 1].set_title(f"BẢN GỐC (Được giữ lại): {os.path.basename(path_orig)}\nLớp: {cls_name}")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.suptitle(f"So sánh {num_show} cặp ảnh trùng lặp tuyệt đối (100% giống nhau)", y=1.02, fontsize=16)
    plt.show()

# Gọi hàm để hiển thị 5 cặp
check_and_visualize_duplicates(RAW_DATASET_DIR, CLASSES, num_pairs=5)

## 2. Kiểm tra và Lọc nhiễu (Noise Estimation)
Đo lường mức độ nhiễu và loại bỏ các ảnh quá nhiễu khỏi danh sách hợp lệ trước khi lưu.Sử dụng thuật toán estimate_sigma từ scikit-image để đo lường phương sai nhiễu (Gaussian noise) trên từng bức ảnh. Những bức ảnh có mức độ nhiễu vượt ngưỡng cho phép (outliers) sẽ bị loại bỏ để đảm bảo chất lượng tập dữ liệu.

In [ ]:
def filter_noisy_images(samples, noise_threshold=15.0):
    print("Đang ước lượng mức độ nhiễu (Sigma)...")
    high_quality_samples = []

    for path, cls in tqdm(samples, desc="Tính toán Noise"):
        img = cv2.imread(path)
        if img is None: continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        sigma = estimate_sigma(gray)

        if sigma <= noise_threshold:
            high_quality_samples.append((path, cls))

    print(f"Tổng số ảnh ban đầu: {len(samples)}")
    print(f"Ảnh đạt tiêu chuẩn giữ lại: {len(high_quality_samples)}")
    print(f"Ảnh quá nhiễu bị loại: {len(samples) - len(high_quality_samples)}")

    return high_quality_samples

# Cập nhật lại danh sách valid_samples
valid_samples = filter_noisy_images(valid_samples, noise_threshold=15.0)

## 3. Định nghĩa hàm xử lý ảnh nâng cao

Hàm preprocess_image thực hiện:

- CLAHE: Tăng cường độ tương phản cục bộ.
- Letterboxing: Thêm padding đen để giữ nguyên tỉ lệ khung hình (Aspect Ratio).

In [ ]:
def preprocess_image(image_path, target_size):
    # Đọc ảnh gốc bằng OpenCV
    img_bgr = cv2.imread(image_path)
    if img_bgr is None: return None, None

    # Chuyển sang RGB để xử lý và hiển thị
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Áp dụng CLAHE (Contrast Limited Adaptive Histogram Equalization)
    # Giúp làm nổi bật điếu thuốc và khói trong các vùng thiếu sáng
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    enhanced_img = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2RGB)

    # 2. Letterbox Resize (Giữ nguyên tỷ lệ ảnh gốc)
    h, w = enhanced_img.shape[:2]
    scale = min(target_size[0]/h, target_size[1]/w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(enhanced_img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)

    # Tạo khung nền đen (Padding)
    delta_w = target_size[1] - new_w
    delta_h = target_size[0] - new_h
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)

    final_img = cv2.copyMakeBorder(resized, top, bottom, left, right,
                                   cv2.BORDER_CONSTANT, value=[0, 0, 0])

    return img_rgb, final_img


### Trực quan hóa so sánh Trước và Sau

In [ ]:
def plot_comparison(num_samples=3):
    if len(valid_samples) == 0:
        print("Không có ảnh để trực quan hóa.")
        return
    num_samples = min(num_samples, len(valid_samples))
    indices = np.random.choice(len(valid_samples), num_samples, replace=False)
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 5 * num_samples))

    for i, idx in enumerate(indices):
        path, cls = valid_samples[idx]
        original, processed = preprocess_image(path, TARGET_SIZE)

        if original is not None:
            # Ảnh gốc
            axes[i, 0].imshow(original)
            axes[i, 0].set_title(f"GỐC: {cls}\nSize: {original.shape[:2]}")
            axes[i, 0].axis('off')

            # Ảnh sau xử lý
            axes[i, 1].imshow(processed)
            axes[i, 1].set_title(f"ĐÃ XỬ LÝ: {cls}\nSize: {processed.shape[:2]} (Letterbox + CLAHE)")
            axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()


### Chạy so sánh trên 3 mẫu ngẫu nhiên

In [ ]:
plot_comparison(num_samples=3)

## 4. Thực thi lưu trữ dữ liệu đã xử lý

Bây giờ chúng ta sẽ áp dụng hàm trên cho toàn bộ tập dữ liệu hợp lệ và lưu vào thư mục processed_dataset.

In [ ]:
for path, cls in tqdm(valid_samples, desc="Processing & Saving"):
    _, processed_img = preprocess_image(path, TARGET_SIZE)

    if processed_img is not None:
        file_name = os.path.basename(path).split('.')[0] + ".jpg"
        save_path = os.path.join(PROCESSED_DATASET_DIR, cls, file_name)
        # Lưu dưới dạng BGR cho OpenCV
        cv2.imwrite(save_path, cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR))

print(f"Hoàn tất! Dữ liệu sạch đã sẵn sàng tại: {PROCESSED_DATASET_DIR}")

## 5. Tăng cường dữ liệu (Data Augmentation)
Sử dụng thư viện Albumentations để tạo ra các biến thể của ảnh. Áp dụng bước này trực tiếp trên thư mục processed_dataset (do toàn bộ ảnh sẽ được dùng làm tập train) để cân bằng dữ liệu và tăng cường khả năng tổng quát hóa của mô hình.

In [ ]:
# Định nghĩa pipeline tăng cường dữ liệu
# Tập trung vào xoay nhẹ, lật ngang và thay đổi độ sáng (phù hợp với ảnh điếu thuốc/khói)
aug_transform = A.Compose([
    A.HorizontalFlip(p=0.5), # Lật ngang 50%
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.7), # Xoay/Dịch chuyển nhẹ
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5), # Đổi sáng/tối
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.2), # Thêm một chút nhiễu có kiểm soát để model "lì đòn" hơn
])

def augment_train_data(train_dir, target_count=2000):
    """
    Hàm này sẽ sinh thêm ảnh cho đến khi mỗi lớp trong tập Train đạt đủ số lượng 'target_count'.
    Giúp giải quyết triệt để vấn đề mất cân bằng dữ liệu (Class Imbalance).
    """
    print(f"Bắt đầu tăng cường dữ liệu tập Train lên {target_count} ảnh/lớp...")

    for cls in CLASSES:
        cls_dir = os.path.join(train_dir, cls)
        existing_images = glob.glob(os.path.join(cls_dir, '*.jpg'))
        current_count = len(existing_images)

        print(f" - Lớp '{cls}': Hiện có {current_count} ảnh.")

        if current_count >= target_count:
            print("   -> Đã đủ số lượng, bỏ qua.")
            continue

        needed_images = target_count - current_count
        print(f"   -> Cần sinh thêm {needed_images} ảnh...")

        # Sinh ảnh mới bằng cách bốc ngẫu nhiên ảnh cũ và đưa qua pipeline Augmentation
        for i in tqdm(range(needed_images), desc=f"Augmenting {cls}", leave=False):
            # Chọn ngẫu nhiên 1 ảnh gốc
            src_img_path = np.random.choice(existing_images)
            img = cv2.imread(src_img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Áp dụng Augmentation
            augmented = aug_transform(image=img)
            aug_img = augmented['image']

            # Lưu ảnh mới
            base_name = os.path.basename(src_img_path).split('.')[0]
            new_file_name = f"{base_name}_aug_{i}.jpg"
            save_path = os.path.join(cls_dir, new_file_name)

            cv2.imwrite(save_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

# Gọi hàm thực thi trực tiếp trên PROCESSED_DATASET_DIR vì toàn bộ dữ liệu này dùng để train
TRAIN_DIR = PROCESSED_DATASET_DIR
augment_train_data(TRAIN_DIR, target_count=2000)

print("\n--- THỐNG KÊ SAU KHI AUGMENTATION ---")
for cls in CLASSES:
    count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    print(f"Lớp '{cls}': {count} ảnh")